In [1]:
import os
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_percentage_error, mean_absolute_error
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

f:\Learning\HUST_DOCUMENTS\MI3390-DoAn2\workspace\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
current = Path.cwd().resolve()
if (current / "src").exists():
    project_root = current
elif (current.parent / "src").exists():
    project_root = current.parent
else:
    project_root = Path(r"F:\Learning\HUST_DOCUMENTS\MI3390-DoAn2\workspace")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: F:\Learning\HUST_DOCUMENTS\MI3390-DoAn2\workspace


In [3]:
from src.utils.spark_utils import get_spark_session
spark = get_spark_session("ML_Experience")
gold_path = str(project_root / "data" / "features" / "gold_layer")
print(f"Đọc dữ liệu từ {gold_path}")
df_spark = spark.read.parquet(gold_path)
df = df_spark.drop("features").toPandas()
print(f"Kích thước DataFrame: {df.shape}") # (9442, 27)
df.head()

f:\Learning\HUST_DOCUMENTS\MI3390-DoAn2\workspace\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Đọc dữ liệu từ F:\Learning\HUST_DOCUMENTS\MI3390-DoAn2\workspace\data\features\gold_layer


f:\Learning\HUST_DOCUMENTS\MI3390-DoAn2\workspace\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Kích thước DataFrame: (9442, 27)


,post_id,target_price,log_price,has_warranty,has_bundle,is_like_new,is_upgraded,is_repaired,is_gpu_inferred,ram_gb_imputed,...,device_category,cpu_vendor,cpu_family,gpu_type,gpu_tier,gpu_family,canonical_resolution,touch_sensitive,origin_type,region_group
0,55737091,3200000.0,14.978662,1.0,1.0,0.0,0.0,0.0,0.0,8.0,...,Mobile Workstation,unknown,unknown,integrated,other,intel_hd,FHD,unknown,xach_tay,hanoi
1,58129676,3400000.0,15.039286,1.0,1.0,1.0,0.0,0.0,0.0,8.0,...,Enterprise Business,unknown,unknown,integrated,unknown,intel_hd,FHD,unknown,xach_tay,hanoi
2,65149087,3400000.0,15.039286,1.0,1.0,1.0,0.0,0.0,0.0,8.0,...,Enterprise Business,unknown,unknown,integrated,unknown,intel_hd,unknown,unknown,xach_tay,hanoi
3,80084278,4500000.0,15.319588,1.0,1.0,0.0,0.0,0.0,0.0,8.0,...,Mobile Workstation,unknown,unknown,integrated,unknown,intel_hd,unknown,unknown,xach_tay,hanoi
4,88208419,8490000.0,15.954400,1.0,1.0,0.0,0.0,0.0,0.0,8.0,...,Mainstream / Thin & Light,intel,core-i5,unknown,unknown,unknown,FHD,unknown,chinh_hang,tier_2


In [4]:
drop_cols = ["post_id", "target_price", "log_price", "features"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df["log_price"]
y_raw = df["target_price"]

In [5]:
X_encoded = pd.get_dummies(X, drop_first=True)

In [6]:
# Train/Test Split (80/20)
X_train, X_test, y_train, y_test, y_raw_train, y_raw_test = train_test_split(
    X_encoded, y, y_raw, test_size=0.2, random_state=42
)

In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [8]:
results = []
def record_result(model_name, tuning_method, model, elapsed_time, best_params):
    y_pred_log = model.predict(X_test)
    y_pred_raw = np.expm1(y_pred_log)
    
    r2 = r2_score(y_test, y_pred_log)
    rmse_log = root_mean_squared_error(y_test, y_pred_log)
    # MAE trên thang đo Log
    mae_log = mean_absolute_error(y_test, y_pred_log)
    # MAE trên giá thực tế (VNĐ)
    mae_vnd = mean_absolute_error(y_raw_test, y_pred_raw)
    # MAPE (% sai số tương đối)
    mape = mean_absolute_percentage_error(y_raw_test, y_pred_raw) * 100
    
    results.append({
        "Model": model_name,
        "Tuning_Method": tuning_method,
        "R2_Score": round(r2, 4),
        "RMSE_Log": round(rmse_log, 4),
        "MAE_Log": round(mae_log, 4),
        "MAE (VNĐ)": f"{mae_vnd:,.0f} đ",  # Định dạng tiền tệ: VD 1,250,000 đ
        "MAPE (%)": round(mape, 2),
        "Time (s)": round(elapsed_time, 2),
        "Best_Params": str(best_params)
    })
    
    print(f"[{model_name}] - {tuning_method} ({elapsed_time:.1f}s) | R2: {r2:.4f} | MAE: {mae_vnd:,.0f} đ | MAPE: {mape:.2f}%")

---
## LINEAR REGRESSION
---

In [9]:
lr_grid_space = {"fit_intercept": [True, False], "positive": [True, False]}
# --- 2.1 Grid Search ---
t0 = time.time()
lr_grid = GridSearchCV(LinearRegression(), lr_grid_space, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X_train, y_train)
record_result("Linear Regression", "Grid Search", lr_grid.best_estimator_, time.time() - t0, lr_grid.best_params_)

[Linear Regression] - Grid Search (6.6s) | R2: 0.7441 | MAE: 3,402,351 đ | MAPE: 38.71%


In [10]:
t0 = time.time()
lr_rand = RandomizedSearchCV(LinearRegression(), lr_grid_space, n_iter=4, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=42).fit(X_train, y_train)
record_result("Linear Regression", "Random Search", lr_rand.best_estimator_, time.time() - t0, lr_rand.best_params_)

[Linear Regression] - Random Search (1.1s) | R2: 0.7441 | MAE: 3,402,351 đ | MAPE: 38.71%


In [11]:
t0 = time.time()
def opt_lr(trial):
    p = {
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "positive": trial.suggest_categorical("positive", [True, False])
    }
    return cross_val_score(LinearRegression(**p), X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1).mean()
study_lr = optuna.create_study(direction="maximize")
study_lr.optimize(opt_lr, n_trials=6)
best_lr = LinearRegression(**study_lr.best_params).fit(X_train, y_train)
record_result("Linear Regression", "Bayesian (Optuna)", best_lr, time.time() - t0, study_lr.best_params)

[Linear Regression] - Bayesian (Optuna) (1.6s) | R2: 0.7441 | MAE: 3,402,351 đ | MAPE: 38.71%


---
## RIDGE REGRESSION
---

In [12]:
ridge_grid_space = {
    "alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
    "fit_intercept": [True, False],
    "solver": ["auto", "lsqr", "saga"]
}
ridge_dist_space = {
    "alpha": np.logspace(-2, 3, 20),
    "fit_intercept": [True, False],
    "solver": ["auto", "lsqr", "saga"]
}

Grid search

In [13]:
t0 = time.time()
ridge_grid = GridSearchCV(Ridge(random_state=42), ridge_grid_space, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X_train, y_train)
record_result("Ridge Regression", "Grid Search", ridge_grid.best_estimator_, time.time() - t0, ridge_grid.best_params_)

[Ridge Regression] - Grid Search (79.1s) | R2: 0.7443 | MAE: 3,398,339 đ | MAPE: 38.70%


f:\Learning\HUST_DOCUMENTS\MI3390-DoAn2\workspace\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.9864726530371968e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Random search

In [14]:
t0 = time.time()
ridge_rand = RandomizedSearchCV(Ridge(random_state=42), ridge_dist_space, n_iter=15, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=42).fit(X_train, y_train)
record_result("Ridge Regression", "Random Search", ridge_rand.best_estimator_, time.time() - t0, ridge_rand.best_params_)

[Ridge Regression] - Random Search (53.2s) | R2: 0.7443 | MAE: 3,397,092 đ | MAPE: 38.69%


f:\Learning\HUST_DOCUMENTS\MI3390-DoAn2\workspace\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 7.579318422878956e-11.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Bayesian optimization

In [15]:
t0 = time.time()
def opt_ridge(trial):
    p = {
        "alpha": trial.suggest_float("alpha", 0.01, 1000.0, log=True),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "solver": trial.suggest_categorical("solver", ["auto", "lsqr", "saga"]),
        "random_state": 42
    }
    return cross_val_score(Ridge(**p), X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1).mean()
study_ridge = optuna.create_study(direction="maximize")
study_ridge.optimize(opt_ridge, n_trials=15)
best_ridge = Ridge(**study_ridge.best_params, random_state=42).fit(X_train, y_train)
record_result("Ridge Regression", "Bayesian (Optuna)", best_ridge, time.time() - t0, study_ridge.best_params)

[Ridge Regression] - Bayesian (Optuna) (47.5s) | R2: 0.7443 | MAE: 3,396,752 đ | MAPE: 38.69%


f:\Learning\HUST_DOCUMENTS\MI3390-DoAn2\workspace\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.3104890356794314e-11.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


---
## DECISION TREE
---

In [16]:
dt_grid_space = {
    "max_depth": [5, 10, 15, 20],
    "min_samples_split": [5, 10, 20],
    "min_samples_leaf": [2, 5, 10]
}
dt_dist_space = {
    "max_depth": [5, 10, 15, 20, 25, None],
    "min_samples_split": [2, 5, 10, 20, 30],
    "min_samples_leaf": [1, 2, 4, 8, 16],
    "max_features": ["sqrt", 0.8, None]
}

Grid search

In [17]:
t0 = time.time()
dt_grid = GridSearchCV(DecisionTreeRegressor(random_state=42), dt_grid_space, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X_train, y_train)
record_result("Decision Tree", "Grid Search", dt_grid.best_estimator_, time.time() - t0, dt_grid.best_params_)

[Decision Tree] - Grid Search (2.2s) | R2: 0.7745 | MAE: 2,592,151 đ | MAPE: 34.24%


Random search

In [18]:
t0 = time.time()
dt_rand = RandomizedSearchCV(DecisionTreeRegressor(random_state=42), dt_dist_space, n_iter=15, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=42).fit(X_train, y_train)
record_result("Decision Tree", "Random Search", dt_rand.best_estimator_, time.time() - t0, dt_rand.best_params_)

[Decision Tree] - Random Search (1.0s) | R2: 0.7815 | MAE: 2,595,171 đ | MAPE: 33.79%


Bayesian optimization

In [19]:
t0 = time.time()
def opt_dt(trial):
    p = {
        "max_depth": trial.suggest_int("max_depth", 5, 25),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", 0.8, None]),
        "random_state": 42
    }
    return cross_val_score(DecisionTreeRegressor(**p), X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1).mean()
study_dt = optuna.create_study(direction="maximize")
study_dt.optimize(opt_dt, n_trials=15)
best_dt = DecisionTreeRegressor(**study_dt.best_params, random_state=42).fit(X_train, y_train)
record_result("Decision Tree", "Bayesian (Optuna)", best_dt, time.time() - t0, study_dt.best_params)

[Decision Tree] - Bayesian (Optuna) (1.3s) | R2: 0.7816 | MAE: 2,685,847 đ | MAPE: 33.66%


---
## RANDOM FOREST
---

In [20]:
rf_grid_space = {
    "n_estimators": [100, 200],
    "max_depth": [10, 15, 20],
    "min_samples_split": [2, 5],
    "max_features": ["sqrt", 0.8]
}
rf_dist_space = {
    "n_estimators": [100, 150, 200, 300],
    "max_depth": [10, 15, 20, 25, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.7, 0.8, None]
}

Grid search

In [21]:
t0 = time.time()
rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), rf_grid_space, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X_train, y_train)
record_result("Random Forest", "Grid Search", rf_grid.best_estimator_, time.time() - t0, rf_grid.best_params_)

[Random Forest] - Grid Search (67.5s) | R2: 0.8379 | MAE: 2,083,235 đ | MAPE: 27.01%


Random search

In [22]:
t0 = time.time()
rf_rand = RandomizedSearchCV(RandomForestRegressor(random_state=42), rf_dist_space, n_iter=15, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=42).fit(X_train, y_train)
record_result("Random Forest", "Random Search", rf_rand.best_estimator_, time.time() - t0, rf_rand.best_params_)

[Random Forest] - Random Search (54.8s) | R2: 0.8349 | MAE: 2,121,177 đ | MAPE: 27.61%


Bayesian optimization

In [23]:
t0 = time.time()
def opt_rf(trial):
    p = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300, step=50),
        "max_depth": trial.suggest_int("max_depth", 10, 25),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 4),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", 0.8]),
        "random_state": 42,
        "n_jobs": -1
    }
    return cross_val_score(RandomForestRegressor(**p), X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error").mean()
study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(opt_rf, n_trials=15)
best_rf = RandomForestRegressor(**study_rf.best_params, random_state=42, n_jobs=-1).fit(X_train, y_train)
record_result("Random Forest", "Bayesian (Optuna)", best_rf, time.time() - t0, study_rf.best_params)

[Random Forest] - Bayesian (Optuna) (42.2s) | R2: 0.8370 | MAE: 2,094,880 đ | MAPE: 27.20%


---
## XGBOOST
---

In [24]:
xgb_grid_space = {
    "n_estimators": [150, 300],
    "learning_rate": [0.05, 0.1],
    "max_depth": [4, 6],
    "subsample": [0.8, 1.0]
}
xgb_dist_space = {
    "n_estimators": [150, 250, 400],
    "learning_rate": [0.03, 0.05, 0.1, 0.15],
    "max_depth": [4, 6, 8],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "reg_alpha": [0.0, 0.1, 1.0],
    "reg_lambda": [0.1, 1.0, 5.0]
}

Grid search

In [25]:
t0 = time.time()
xgb_grid = GridSearchCV(XGBRegressor(random_state=42, n_jobs=-1), xgb_grid_space, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X_train, y_train)
record_result("XGBoost", "Grid Search", xgb_grid.best_estimator_, time.time() - t0, xgb_grid.best_params_)

[XGBoost] - Grid Search (11.4s) | R2: 0.8475 | MAE: 2,065,580 đ | MAPE: 26.62%


Random search

In [26]:
t0 = time.time()
xgb_rand = RandomizedSearchCV(XGBRegressor(random_state=42, n_jobs=-1), xgb_dist_space, n_iter=15, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=42).fit(X_train, y_train)
record_result("XGBoost", "Random Search", xgb_rand.best_estimator_, time.time() - t0, xgb_rand.best_params_)

[XGBoost] - Random Search (13.6s) | R2: 0.8557 | MAE: 1,970,025 đ | MAPE: 25.73%


Bayeisan optimization

In [27]:
t0 = time.time()
def opt_xgb(trial):
    p = {
        "n_estimators": trial.suggest_int("n_estimators", 150, 400, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
        "random_state": 42,
        "n_jobs": -1
    }
    return cross_val_score(XGBRegressor(**p), X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error").mean()
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(opt_xgb, n_trials=15)
best_xgb = XGBRegressor(**study_xgb.best_params, random_state=42, n_jobs=-1).fit(X_train, y_train)
record_result("XGBoost", "Bayesian (Optuna)", best_xgb, time.time() - t0, study_xgb.best_params)

[XGBoost] - Bayesian (Optuna) (37.0s) | R2: 0.8490 | MAE: 2,037,935 đ | MAPE: 26.28%


---
## CATBOOST
---

In [28]:
cat_grid_space = {
    "iterations": [200, 350],
    "learning_rate": [0.05, 0.1],
    "depth": [4, 6]
}
cat_dist_space = {
    "iterations": [200, 300, 400],
    "learning_rate": [0.03, 0.06, 0.1, 0.15],
    "depth": [4, 6, 8],
    "l2_leaf_reg": [1.0, 3.0, 5.0, 9.0]
}

Grid search

In [29]:
t0 = time.time()
cat_grid = GridSearchCV(CatBoostRegressor(verbose=0, random_seed=42, thread_count=-1), cat_grid_space, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=1).fit(X_train, y_train)
record_result("CatBoost", "Grid Search", cat_grid.best_estimator_, time.time() - t0, cat_grid.best_params_)

[CatBoost] - Grid Search (23.6s) | R2: 0.8460 | MAE: 2,123,907 đ | MAPE: 27.05%


Random search

In [30]:
t0 = time.time()
cat_rand = RandomizedSearchCV(CatBoostRegressor(verbose=0, random_seed=42, thread_count=-1), cat_dist_space, n_iter=15, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=42).fit(X_train, y_train)
record_result("CatBoost", "Random Search", cat_rand.best_estimator_, time.time() - t0, cat_rand.best_params_)

[CatBoost] - Random Search (28.4s) | R2: 0.8470 | MAE: 2,064,354 đ | MAPE: 26.49%


Bayeisan optimization

In [31]:
t0 = time.time()
def opt_cat(trial):
    p = {
        "iterations": trial.suggest_int("iterations", 200, 400, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "verbose": 0,
        "random_seed": 42,
        "thread_count": -1
    }
    return cross_val_score(CatBoostRegressor(**p), X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error").mean()
study_cat = optuna.create_study(direction="maximize")
study_cat.optimize(opt_cat, n_trials=15)
best_cat = CatBoostRegressor(**study_cat.best_params, verbose=0, random_seed=42, thread_count=-1).fit(X_train, y_train)
record_result("CatBoost", "Bayesian (Optuna)", best_cat, time.time() - t0, study_cat.best_params)

[CatBoost] - Bayesian (Optuna) (73.0s) | R2: 0.8479 | MAE: 2,059,932 đ | MAPE: 26.43%


---
## SYNTHETIC
---

In [32]:
df_summary = pd.DataFrame(results).sort_values(by=["Model", "R2_Score"], ascending=[True, False])
print("\n" + "="*80)
print("BẢNG TỔNG HỢP ĐỐI SÁNH 18 KỊCH BẢN THỰC NGHIỆM")
print("="*80)
df_summary[["Model", "Tuning_Method", "R2_Score", "RMSE_Log", "MAPE (%)", "Time (s)"]]


BẢNG TỔNG HỢP ĐỐI SÁNH 18 KỊCH BẢN THỰC NGHIỆM


,Model,Tuning_Method,R2_Score,RMSE_Log,MAPE (%),Time (s)
17,CatBoost,Bayesian (Optuna),0.8479,0.3591,26.43,72.97
16,CatBoost,Random Search,0.8470,0.3601,26.49,28.43
15,CatBoost,Grid Search,0.8460,0.3614,27.05,23.55
8,Decision Tree,Bayesian (Optuna),0.7816,0.4303,33.66,1.33
7,Decision Tree,Random Search,0.7815,0.4303,33.79,1.00
6,Decision Tree,Grid Search,0.7745,0.4372,34.24,2.21
0,Linear Regression,Grid Search,0.7441,0.4657,38.71,6.61
1,Linear Regression,Random Search,0.7441,0.4657,38.71,1.10
2,Linear Regression,Bayesian (Optuna),0.7441,0.4657,38.71,1.59
9,Random Forest,Grid Search,0.8379,0.3707,27.01,67.49
